# TALLER PRÁCTICO: APLICACIÓN DE ALGORITMOS DE CLASIFICACIÓN Y CLUSTERING

## Objetivos
- Implementar algoritmos de clasificación y clustering en Python
- Evaluar y comparar el rendimiento de diferentes métodos
- Aplicar técnicas de visualización para interpretar resultados
- Desarrollar habilidades prácticas en el uso de las bibliotecas scikit-learn, pandas y matplotlib

## Instrucciones
Este taller se realizará usando Google Colab. Trabajaremos con el conjunto de datos "Iris", un dataset clásico en el campo del aprendizaje automático. Este dataset contiene mediciones de diferentes especies de flores iris.

### Parte 1: Preparación del entorno

1. Accede a [Google Colab](https://colab.research.google.com/)
2. Instala y carga las bibliotecas necesarias

In [ ]:
# Importar bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, silhouette_score

### Parte 2: Carga y exploración de datos

1. Carga el conjunto de datos Iris
2. Realiza un análisis exploratorio básico
3. Visualiza las relaciones entre características
 - Genere una matriz de gráficos de dispersión que relacione cada característica, utilizando el color para diferenciar las especies.
 - Genere un gráfico de distribución de cada característica por especie.
 - Genere un mapa de Calor de la correlación entre las características.
 - ¿Existe alguna característica que le permita diferenciar claramente entre especies?

In [ ]:
# Cargamos el dataset Iris desde scikit-learn
iris = datasets.load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Creamos un DataFrame para facilitar el análisis
iris_df = pd.DataFrame(X, columns=feature_names)
iris_df['species'] = pd.Categorical.from_codes(y, target_names)

In [ ]:
# 2. Mostramos información básica del dataset
print("Dimensiones del dataset:", iris_df.shape)
print("\nPrimeras 5 filas:")
print(iris_df.head())

print("\nInformación del dataset:")
print(iris_df.info())

print("\nEstadísticas descriptivas:")
print(iris_df.describe())

print("\nDistribución de clases:")
print(iris_df['species'].value_counts())

In [ ]:
# Visualización: matriz de dispersión para ver relaciones entre variables
# Este tipo de visualización es muy útil para datasets con múltiples características
plt.figure(figsize=(15, 10))
sns.pairplot(iris_df, hue='species', height=2.5)
plt.suptitle('Matriz de dispersión del conjunto de datos Iris', y=1.02, fontsize=16)
plt.show()

In [ ]:
# Visualización: distribución de cada característica por especie
plt.figure(figsize=(15, 10))
for i, feature in enumerate(feature_names):
    plt.subplot(2, 2, i+1)
    for species in target_names:
        subset = iris_df[iris_df['species'] == species]
        sns.histplot(subset[feature], kde=True, label=species)
    plt.title(f'Distribución de {feature}')
    plt.xlabel(feature)
    plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualización: mapa de calor de correlación
plt.figure(figsize=(10, 8))
correlation = iris_df.drop('species', axis=1).corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Mapa de calor de correlación entre características')
plt.show()

### Parte 3: Clasificación supervisada

1. Divide los datos en conjuntos de entrenamiento y prueba
2. Estandariza los datos para mejorar el rendimiento de K-NN y Regresión Logística
3. Implementa y evalúa los siguientes algoritmos:
   - K-Nearest Neighbors (K-NN)
   - Regresión Logística
   - Naive Bayes
   - Árbol de Decisión
4. Compara los resultados utilizando métricas como precisión, matriz de confusión y reporte de clasificación

In [ ]:
# Parte 3: Clasificación supervisada
# Separamos los datos en conjuntos de entrenamiento y prueba (70% - 30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Estandarizamos los datos para mejorar el rendimiento de algunos algoritmos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# 1. K-Nearest Neighbors (K-NN)
# Implementamos el algoritmo con diferentes valores de k para encontrar el óptimo
k_values = list(range(1, 21))
k_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    k_accuracies.append(accuracy_score(y_test, y_pred))

# Visualizamos el efecto del parámetro k
plt.figure(figsize=(10, 6))
plt.plot(k_values, k_accuracies, marker='o', linestyle='-')
plt.title('Precisión de K-NN según el número de vecinos (k)')
plt.xlabel('Número de vecinos (k)')
plt.ylabel('Precisión')
plt.grid(True)
plt.xticks(k_values)
plt.show()

In [ ]:
# Seleccionamos el mejor valor de k
best_k = k_values[np.argmax(k_accuracies)]
print(f"Mejor valor de k: {best_k} con precisión: {max(k_accuracies):.4f}")

# Evaluamos K-NN con el mejor valor de k
knn_best = KNeighborsClassifier(n_neighbors=best_k)

In [ ]:
# 2. Regresión Logística
# Implementamos regresión logística multinomial (para múltiples clases)
logreg = LogisticRegression(max_iter=200, multi_class='multinomial', solver='lbfgs')

In [ ]:
# 3. Naive Bayes
# Implementamos Naive Bayes Gaussiano (adecuado para características continuas)
nb = GaussianNB()

In [ ]:
# 4. Árbol de Decisión
# Implementamos un árbol de decisión y visualizamos su estructura
dt = DecisionTreeClassifier(random_state=42)

In [ ]:
# Función auxiliar para evaluar y visualizar resultados de los clasificadores
def evaluate_classifier(clf, X_train, X_test, y_train, y_test, name):
    """
    Entrena, evalúa y visualiza resultados de un clasificador.

    Parámetros:
    - clf: Clasificador a evaluar
    - X_train, X_test, y_train, y_test: Conjuntos de entrenamiento y prueba
    - name: Nombre del clasificador para mostrar en gráficos
    """
    # Entrenamos el modelo
    clf.fit(X_train, y_train)

    # Hacemos predicciones
    y_pred = clf.predict(X_test)

    # Evaluamos el modelo
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, target_names=target_names)

    # Mostramos resultados
    print(f"=== {name} ===")
    print(f"Precisión: {accuracy:.4f}")
    print("\nMatriz de confusión:")
    print(conf_matrix)
    print("\nReporte de clasificación:")
    print(class_report)

    # Visualizamos la matriz de confusión
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Matriz de confusión - {name}')
    plt.xlabel('Predicción')
    plt.ylabel('Valor real')
    plt.show()

    return accuracy, clf

In [ ]:
knn_accuracy, knn_clf = evaluate_classifier(knn_best, X_train_scaled, X_test_scaled,
                                           y_train, y_test, "K-Nearest Neighbors")

logreg_accuracy, logreg_clf = evaluate_classifier(logreg, X_train_scaled, X_test_scaled,
                                                 y_train, y_test, "Regresión Logística")

nb_accuracy, nb_clf = evaluate_classifier(nb, X_train, X_test, y_train, y_test, "Naive Bayes")

dt_accuracy, dt_clf = evaluate_classifier(dt, X_train, X_test, y_train, y_test, "Árbol de Decisión")

In [ ]:
# Visualización del árbol de decisión
plt.figure(figsize=(20, 10))
plot_tree(dt_clf, feature_names=feature_names, class_names=target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Visualización del Árbol de Decisión')
plt.show()

In [ ]:
# Comparativa de todos los clasificadores
classifiers = ['K-NN', 'Regresión Logística', 'Naive Bayes', 'Árbol de Decisión']
accuracies = [knn_accuracy, logreg_accuracy, nb_accuracy, dt_accuracy]

plt.figure(figsize=(10, 6))
bars = plt.bar(classifiers, accuracies, color=['skyblue', 'lightgreen', 'salmon', 'purple'])
plt.title('Comparación de precisión entre clasificadores')
plt.xlabel('Clasificador')
plt.ylabel('Precisión')
plt.ylim([0, 1.05])
plt.grid(axis='y', alpha=0.3)

# Añadimos etiquetas con los valores de precisión sobre cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.4f}', ha='center', va='bottom', fontsize=12)
plt.show()

### Parte 4: Clustering no supervisado

1. Aplica el algoritmo K-means al conjunto de datos. Recuerde no usar etiquetas y seleccionar un valor de K acorde al número de grupos que espera obtener.
2. Evalúa el rendimiento usando métricas como silueta e inercia.
3. Visualiza los clusters resultantes comparando con grupos reales.

In [ ]:
# Parte 4: Clustering no supervisado
# Aplicamos K-means al dataset Iris sin utilizar las etiquetas

# Función para evaluar el rendimiento de K-means con diferentes valores de k
def evaluate_kmeans(X, y_true, k_range):
    """
    Evalúa K-means con diferentes valores de k utilizando
    la puntuación de silueta y la inercia.

    Parámetros:
    - X: Datos para clustering
    - y_true: Etiquetas reales (solo para visualización)
    - k_range: Rango de valores de k a evaluar
    """
    inertias = []
    silhouette_scores = []

    for k in k_range:
        # Entrenamos K-means
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        y_kmeans = kmeans.predict(X)

        # Calculamos métricas
        inertias.append(kmeans.inertia_)

        # Silhouette solo se puede calcular para k >= 2
        if k > 1:
            silhouette_scores.append(silhouette_score(X, y_kmeans))
        else:
            silhouette_scores.append(0)  # Para k=1, la puntuación de silueta no está definida

    # Visualizamos los resultados
    plt.figure(figsize=(15, 6))

    # Gráfico de codo para inertia
    plt.subplot(1, 2, 1)
    plt.plot(k_range, inertias, marker='o', linestyle='-')
    plt.title('Método del codo para K-means')
    plt.xlabel('Número de clusters (k)')
    plt.ylabel('Inercia')
    plt.grid(True)
    plt.xticks(k_range)

    # Gráfico de puntuación de silueta
    plt.subplot(1, 2, 2)
    plt.plot(k_range[1:], silhouette_scores[1:], marker='o', linestyle='-')
    plt.title('Puntuación de silueta para K-means')
    plt.xlabel('Número de clusters (k)')
    plt.ylabel('Puntuación de silueta')
    plt.grid(True)
    plt.xticks(k_range[1:])

    plt.tight_layout()
    plt.show()

    # Determinamos el mejor k basado en la puntuación de silueta
    best_k = k_range[1:][np.argmax(silhouette_scores[1:])]
    print(f"Mejor valor de k según puntuación de silueta: {best_k}")

    return best_k

In [ ]:
# Evaluamos K-means con diferentes valores de k
k_range = range(1, 11)
best_k = evaluate_kmeans(X, y, k_range)

# Implementamos K-means con el mejor valor de k
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X)

# Visualizamos los clusters en 2D
# Para ello seleccionamos las dos características más discriminativas
# En este caso, podemos usar longitud y anchura de pétalo (índices 2 y 3)
plt.figure(figsize=(15, 6))

In [ ]:
# Gráfico con los clusters obtenidos vs. etiquetas reales
# Creamos dos subgráficos para comparar
plt.subplot(1, 2, 1)
plt.scatter(X[:, 2], X[:, 3], c=y_kmeans, cmap='viridis',
            s=50, edgecolor='k', alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 2], kmeans.cluster_centers_[:, 3],
            c='red', marker='X', s=200, alpha=0.8, label='Centroides')
plt.title(f'Clusters K-means (k={best_k})')
plt.xlabel(feature_names[2])
plt.ylabel(feature_names[3])
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 2], X[:, 3], c=y, cmap='viridis',
            s=50, edgecolor='k', alpha=0.8)
plt.title('Clases reales')
plt.xlabel(feature_names[2])
plt.ylabel(feature_names[3])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 3D para mejor comprensión
from mpl_toolkits.mplot3d import Axes3D

# Seleccionamos tres características para visualización 3D
fig = plt.figure(figsize=(12, 5))

# Gráfico 3D con clusters K-means
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X[:, 0], X[:, 2], X[:, 3], c=y_kmeans, cmap='viridis',
            s=50, alpha=0.6, edgecolor='k')
ax1.set_title(f'Clusters K-means 3D (k={best_k})')
ax1.set_xlabel(feature_names[0])
ax1.set_ylabel(feature_names[2])
ax1.set_zlabel(feature_names[3])

# Gráfico 3D con etiquetas reales
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X[:, 0], X[:, 2], X[:, 3], c=y, cmap='viridis',
            s=50, alpha=0.6, edgecolor='k')
ax2.set_title('Clases reales 3D')
ax2.set_xlabel(feature_names[0])
ax2.set_ylabel(feature_names[2])
ax2.set_zlabel(feature_names[3])

plt.tight_layout()
plt.show()

In [ ]:
# Evaluamos la calidad del clustering comparando con las etiquetas reales
# Para esto, podemos usar una tabla de contingencia
contingency_table = pd.crosstab(pd.Series(y, name='Real'),
                               pd.Series(y_kmeans, name='K-means'))
print("Tabla de contingencia (clases reales vs. clusters):")
print(contingency_table)

# Calculamos la puntuación de silueta para el mejor k
silhouette = silhouette_score(X, y_kmeans)
print(f"Puntuación de silueta para k={best_k}: {silhouette:.4f}")

In [ ]:
# Resumen de resultados
print("\n=== RESUMEN DE RESULTADOS ===")
print("\nPrecisión de los clasificadores:")
for clf_name, acc in zip(classifiers, accuracies):
    print(f"- {clf_name}: {acc:.4f}")

print(f"\nClusterización K-means (k={best_k}):")
print(f"- Puntuación de silueta: {silhouette:.4f}")